<a href="https://colab.research.google.com/github/ayachraiet88-crypto/customer-churn/blob/main/Breast_Cancer_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
from sklearn.datasets import load_breast_cancer

In [ ]:
data=load_breast_cancer()

In [ ]:
X=pd.DataFrame(data.data,columns=data.feature_names)
y=data.target

In [ ]:
print(X.shape)

(569, 30)


In [ ]:
print(X.head())

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst radius  worst texture  worst perimeter  \
0           

In [ ]:
print(pd.Series(y).value_counts())

1    357
0    212
Name: count, dtype: int64


In [ ]:
print(data.target_names)

['malignant' 'benign']


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
parametres_knn={
    "n_neighbors":[3,5,7,9,11],
    "weights":["uniform","distance"]
}
grid_knn_cancer=GridSearchCV(KNeighborsClassifier(),parametres_knn,cv=5,scoring="recall_macro")
grid_knn_cancer.fit(X_train_scaled,y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [3, 5, 7, 9, 11],
                         'weights': ['uniform', 'distance']},
             scoring='recall_macro')

In [ ]:
print("meilleur parametres:",grid_knn_cancer.best_params_)

meilleur parametres: {'n_neighbors': 7, 'weights': 'uniform'}


In [ ]:
y_pred=grid_knn_cancer.predict(X_test_scaled)
print(classification_report(y_test,y_pred,target_names=["malignant","benign"]))

              precision    recall  f1-score   support

   malignant       1.00      0.93      0.96        42
      benign       0.96      1.00      0.98        72

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



In [ ]:
import numpy as np

In [ ]:
X_train_scaled_df=pd.DataFrame(X_train_scaled,columns=X.columns)
X_train_scaled_df["target"]=y_train

In [ ]:
difference_zscore=X_train_scaled_df.groupby("target").mean().diff().iloc[-1].abs().sort_values(ascending=False)
print(difference_zscore.head(10))

worst concave points    1.649920
worst perimeter         1.619329
mean concave points     1.612157
worst radius            1.604887
mean perimeter          1.540274
worst area              1.525458
mean radius             1.513591
mean area               1.474697
mean concavity          1.430551
worst concavity         1.381593
Name: 1, dtype: float64


In [ ]:
moyennes = X_train.mean()
moyennes.to_csv("moyennes_cancer.csv")

In [ ]:

import joblib

joblib.dump(grid_knn_cancer, "cancer_model.pkl")
joblib.dump(scaler, "cancer_scaler.pkl")

['cancer_scaler.pkl']

In [ ]:
%%writefile app_cancer.py
import joblib

import pandas as pd
import gradio as gr
import numpy as np

model = joblib.load("cancer_model.pkl")
scaler = joblib.load("cancer_scaler.pkl")
moyennes = pd.read_csv("moyennes_cancer.csv", index_col=0).squeeze()
def predire_cancer(worst_concave_points, worst_perimeter, mean_concave_points, worst_radius, mean_perimeter):
    donnees = moyennes.copy()
    donnees["worst concave points"] = worst_concave_points
    donnees["worst perimeter"] = worst_perimeter
    donnees["mean concave points"] = mean_concave_points
    donnees["worst radius"] = worst_radius
    donnees["mean perimeter"] = mean_perimeter

    donnees_array = np.array([donnees.values])
    donnees_scaled = scaler.transform(donnees_array)

    prediction = model.predict(donnees_scaled)[0]
    probabilites = model.predict_proba(donnees_scaled)[0]

    return {"Malignant (malin)": float(probabilites[0]), "Benign (bénin)": float(probabilites[1])}

interface = gr.Interface(
    fn=predire_cancer,
    inputs=[
        gr.Slider(0.0, 0.3, label="Worst concave points"),
        gr.Slider(50.0, 250.0, label="Worst perimeter"),
        gr.Slider(0.0, 0.2, label="Mean concave points"),
        gr.Slider(7.0, 40.0, label="Worst radius"),
        gr.Slider(40.0, 200.0, label="Mean perimeter")
    ],
    outputs=gr.Label(num_top_classes=2),
    title="🩺 Breast Cancer Predictor",
    description="Démo éducative — ne remplace jamais un diagnostic médical professionnel."
)

interface.launch()

interface.launch(share=True)

Overwriting app_cancer.py


In [ ]:
%run app_cancer.py

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://728eb6f7e97da7e0da.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://728eb6f7e97da7e0da.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
